In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier

In [2]:
artificial_test_data = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_test_data.csv")
artificial_train_data = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_train_data.csv")
artificial_train_labels = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_train_labels.csv")

In [3]:
X_train = artificial_train_data
y_train = artificial_train_labels
X_test = artificial_test_data

In [5]:
print("Wymiary zbiorów danych:")
print("X_train: ", X_train.shape)
print("y_train: ", y_train.shape)
print("X_test: ", X_test.shape)

Wymiary zbiorów danych:
X_train:  (1500, 100)
y_train:  (1500, 1)
X_test:  (500, 100)


In [6]:
print(y_train.value_counts(normalize=True))
print(y_train.value_counts())

Class
1        0.503333
2        0.496667
Name: proportion, dtype: float64
Class
1        755
2        745
Name: count, dtype: int64


In [7]:
print("Numery wierszy zawierające braki danych w poszczególnych zabiorach danych:")
print(X_train[X_train.isna().any(axis=1)].index)
print(y_train[y_train.isna().any(axis=1)].index)
print(X_test[X_test.isna().any(axis=1)].index)

Numery wierszy zawierające braki danych w poszczególnych zabiorach danych:
Index([], dtype='int64')
Index([], dtype='int64')
Index([], dtype='int64')


In [8]:
macierz_korelacji = X_train.corr().abs()
pary = macierz_korelacji.unstack()
posortowane_pary = pary.sort_values(kind="quicksort", ascending=False)
silnie_korelujace_pary = posortowane_pary[(posortowane_pary < 1.0) & (posortowane_pary > 0.5)]
print("Najsilniej skorelowane pary kolumn:")
print(silnie_korelujace_pary.iloc[::2].head(15))

Najsilniej skorelowane pary kolumn:
V9   V65    0.991098
V73  V50    0.989701
     V43    0.989635
V57  V90    0.989475
V8   V83    0.989457
V57  V87    0.989100
V42  V67    0.989044
V87  V90    0.988920
V50  V43    0.988787
V58  V10    0.988740
V91  V3     0.988694
V22  V64    0.988669
V9   V91    0.734608
V65  V91    0.733312
V3   V9     0.729124
dtype: float64


In [9]:
class Usun_mocno_skorelowane:
    def __init__(self, granica=0.95):
        self.granica = granica
        self.niepotrzebne_kolumny = []

    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            df = pd.DataFrame(X)
        else: 
            df = X
        macierz_korelacji = df.corr().abs()
        gorny_trojkat_mk = macierz_korelacji.where(np.triu(np.ones(macierz_korelacji.shape), k=1).astype(bool))
        self.niepotrzebne_kolumny = [col for col in gorny_trojkat_mk.columns if any(gorny_trojkat_mk[col] > self.granica)]
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray): 
            df = pd.DataFrame(X)
            return df.drop(columns=self.niepotrzebne_kolumny, axis=1).values
        else:
            return X.drop(columns=self.niepotrzebne_kolumny, axis=1)

In [10]:
tree_params = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [None, 5, 7, 9, 11, 13, 15],
    "model__min_samples_leaf": np.arange(3, 12, 2)}

logreg_params = [
   {"model__solver": ["liblinear"],
    "model__penalty": ["l1", "l2"],
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
   {"model__solver": ["lbfgs"],
    "model__penalty" : [None, "l2"],
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]},
   {"model__solver": ["saga"],
    "model__penalty" : ["elasticnet"],
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000],
    "model__l1_ratio": [0.2, 0.5, 0.8]}]

svc_params = {
    "model__kernel": ["rbf"],
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100]}

knn_params = {
    "model__n_neighbors": np.arange(10, 41, 5),
    "model__weights": ["distance", "uniform"],
    "model__metric": ["euclidean", "manhattan", "minkowski"]}

nb_params = {
    "model__alpha": [0.01, 0.1, 0.5, 1, 10],
    "model__binarize": [-0.5, 0, 0.5]}

lda_params = {
    "model__solver": ["svd"],
    "model__tol": [1e-4, 1e-5]}

qda_params = {
    "model__reg_param": [0, 0.1, 0.5, 0.9],
    "model__tol": [1e-4, 1e-5]}

bc_params = {
    "model__estimator": [DecisionTreeClassifier()],
    "model__n_estimators": np.arange(50, 201, 25)}

et_params = {
    "model__n_estimators": np.arange(50, 401, 50),
    "model__max_features": [None, "sqrt"]}

gb_params = {
    "model__learning_rate": [0.05, 0.1, 0.2, 0.3, 0.4],
    "model__n_estimators": np.arange(50, 301, 50)}

In [11]:
models = [
    #("Decision Tree", DecisionTreeClassifier(class_weight="balanced", random_state=1000), tree_params),
    #("Logistic Regression", LogisticRegression(class_weight="balanced", max_iter=500, random_state=1000), logreg_params),
    #("SVM", SVC(class_weight="balanced", probability=True, random_state=1000), svc_params),
    #("LDA", LinearDiscriminantAnalysis(), lda_params),
    #("QDA", QuadraticDiscriminantAnalysis(), qda_params),
    #("K-Nearest Neighbors", KNeighborsClassifier(), knn_params),
    #("Naive Bayes", BernoulliNB(), nb_params),
    #("Bagging Classifier", BaggingClassifier(random_state=1000), bc_params),
    ("Extra Tree", ExtraTreesClassifier(random_state=1000, class_weight="balanced"), et_params),
    #("Gradient Boosting", GradientBoostingClassifier(random_state=1000), gb_params)
    ]

results = []
best_project_model = None
best_project_score = 0

In [12]:
cv = StratifiedKFold(n_splits=7, shuffle=True, random_state=1000)

In [13]:
i = 1
for name, model, params in models:
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("const", VarianceThreshold(threshold=0.05)),
        #("corr_selector", Usun_mocno_skorelowane(granica=0.95)),
        ("model", model)])
    
    grid_pipeline = GridSearchCV(pipeline, param_grid=params, cv=cv, scoring="balanced_accuracy", n_jobs=-1)
    
    grid_pipeline.fit(X_train, y_train)
    best_params = grid_pipeline.best_params_
    score = grid_pipeline.best_score_
          
    results.append({"Model": name, "Best Params": best_params, "BA Score": score})
    
    if score > best_project_score:
            best_project_score = score
            best_project_model = grid_pipeline.best_estimator_

    print(i)
    i += 1

c:\Users\przem\anaconda\Lib\site-packages\sklearn\base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


1


In [14]:
df_results = pd.DataFrame(results)
df_results

,Model,Best Params,BA Score
0,Extra Tree,"{'model__max_features': None, 'model__n_estima...",0.874866


In [15]:
winner = df_results.sort_values(by="BA Score", ascending=False).iloc[0]

print("Najlepszą moc predykcyjną ma model:", winner["Model"])
print("Z parametrami:", winner["Best Params"])
print("Miara BA dla tego modelu wynosi:", winner["BA Score"])

Najlepszą moc predykcyjną ma model: Extra Tree
Z parametrami: {'model__max_features': None, 'model__n_estimators': 200}
Miara BA dla tego modelu wynosi: 0.8748656489575197


In [16]:
y_test_pred = pd.DataFrame(best_project_model.predict(X_test), columns=["Test_Class"])
y_test_pred

,Test_Class
0,2
1,1
2,2
3,1
4,1
...,...
495,2
496,2
497,1
498,2


In [17]:
print(y_test_pred.value_counts(normalize=True))
print(y_test_pred.value_counts())

Test_Class
2             0.526
1             0.474
Name: proportion, dtype: float64
Test_Class
2             263
1             237
Name: count, dtype: int64


In [18]:
probs = best_project_model.predict_proba(X_test)[:, 0]
nazwa_pliku = f"333063_artifical_prediction.txt"
np.savetxt(nazwa_pliku, probs, fmt="%.18f")